# TT/MPS基礎 08 — SVD Center Move と Schmidt 形

## 今回の位置づけ

前回の Notebook 07 では、3階 TT/MPS

$$
X
=
G_1^{[L]}
G_2^{[C]}
G_3^{[R]}
$$

を出発点として、QR による orthogonality center の

$$
2\rightarrow3,
\qquad
2\rightarrow1
$$

の移動を確認しました。

QR による center move では、

- 全テンソル $X$ は不変
- center の左側を左直交にできる
- center の右側を右直交にできる
- center の位置にノルムを局所化できる

ことを確認済みです。

今回は同じ center move を **SVD** で行い、その中に現れる

$$
\Sigma
=
\operatorname{diag}
(\sigma_1,\ldots,\sigma_\rho)
$$

を明示的に取り出します。

そして、切断

$$
(i_1,i_2)\mid i_3
$$

に対して

$$
|X\rangle
=
\sum_{\beta=1}^{\rho}
\sigma_\beta
|L_\beta\rangle
\otimes
|R_\beta\rangle
$$

という Schmidt 形までつなげます。

### 今回やること

1. QR center move と SVD center move の共通点・相違点を整理する
2. $G_2^{[C]\langle L\rangle}$ に reduced SVD を適用する
3. $U$ と $\Sigma V^T$ による exact $2\rightarrow3$ center move を行う
4. 再構成不変性・左直交性・center norm を確認する
5. $V^T$ によって右 bond 基底を回転する
6. $\Sigma$ を bond 上に明示した形を作る
7. 左右 Schmidt 状態を構成し、正規直交性を確認する
8. 
   $$
   \|X\|_F^2
   =
   \sum_{\beta=1}^{\rho}\sigma_\beta^2
   $$
   を確認する
9. QR move と exact SVD move を比較する

### 今回はまだ扱わないもの

- 特異値の打ち切り
- truncated SVD
- rank-$k$ 近似
- Eckart–Young–Mirsky 定理の適用
- entanglement entropy
- TT rounding
- TT-matrix / MPO
- DMRG / ALS / 中心コア最適化

**今回の SVD では、返された特異値を一つも捨てません。**


## 1. 出発点：第2サイト中心の mixed-canonical form

出発点は、

$$
X
=
G_1^{[L]}
G_2^{[C]}
G_3^{[R]}.
$$

shape は、

$$
G_1^{[L]}
\in
\mathbb{R}^{1\times n_1\times r_1},
$$

$$
G_2^{[C]}
\in
\mathbb{R}^{r_1\times n_2\times r_2},
$$

$$
G_3^{[R]}
\in
\mathbb{R}^{r_2\times n_3\times1}.
$$

左右はそれぞれ直交化されており、

$$
\left(
G_1^{[L]\langle L\rangle}
\right)^T
G_1^{[L]\langle L\rangle}
=
I_{r_1},
$$

$$
G_3^{[R]\langle R\rangle}
\left(
G_3^{[R]\langle R\rangle}
\right)^T
=
I_{r_2}.
$$

そのため中心コアは、左・右の正規直交基底の間にある係数テンソルとして扱えます。

今回 SVD を行う対象は、中心コアの**左展開**

$$
A
=
G_2^{[C]\langle L\rangle}
\in
\mathbb{R}^{(r_1n_2)\times r_2}
$$

です。

添字で書けば、

$$
A_{(\alpha_1,i_2),\alpha_2}
=
G_2^{[C]}(\alpha_1,i_2,\alpha_2).
$$


## 2. QR と SVD は何が同じで、何が違うか

同じ行列

$$
A
=
G_2^{[C]\langle L\rangle}
$$

に対して、QR では

$$
A=QR
$$

と分解しました。

SVD では、

$$
A
=
U\Sigma V^T
$$

と分解します。

center を右へ移動するという観点では、

$$
\boxed{
Q
\longleftrightarrow
U
}
$$

$$
\boxed{
R
\longleftrightarrow
\Sigma V^T
}
$$

という対応があります。

どちらも、

1. 左側に直交因子を残す
2. 残りの係数を右隣へ渡す
3. center を一つ右へ移す

という操作です。

### 違い

QR の $R$ は一般の上三角行列です。

一方、SVD では

$$
\Sigma
=
\operatorname{diag}
(\sigma_1,\ldots,\sigma_\rho)
$$

という**対角な重み**が現れます。

したがって、

$$
\boxed{
\text{QR：直交化と center 移動}
}
$$

に対して、

$$
\boxed{
\text{SVD：直交化と center 移動}
+
\text{ bond ごとの特異値}
}
$$

という違いがあります。

今回の中心テーマは、この $\Sigma$ が Schmidt 形へどうつながるかです。


## 3. 今回の数値検証用セットアップ

Notebook 07 と同じく、小さい3階 TT を作り、

$$
X
=
G_1^{[L]}
G_2^{[C]}
G_3^{[R]}
$$

という第2サイト中心の mixed-canonical form までをセットアップとして用意します。

この部分は既習内容なので、今回の演習対象にはしません。

以後は、

- `G1_left`
- `G2_center`
- `G3_right`
- `X_center2`

を出発点にします。


In [1]:
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)


def reconstruct_tt3(
    G1: torch.Tensor,
    G2: torch.Tensor,
    G3: torch.Tensor,
) -> torch.Tensor:
    """3個のTTコアから3階テンソルを再構成する。"""
    return torch.einsum("aib,bjc,ckd->ijk", G1, G2, G3)


# 小さい3階TT
n1, n2, n3 = 4, 3, 5
r1, r2 = 2, 3

G1 = torch.randn(1, n1, r1)
G2 = torch.randn(r1, n2, r2)
G3 = torch.randn(r2, n3, 1)

X_before = reconstruct_tt3(G1, G2, G3)

# --- Notebook 06 / 07 までに学習済みの mixed-canonical 化 ---

# 第1コアを左直交化
A1 = G1.squeeze(0)
Q1, R1 = torch.linalg.qr(A1, mode="reduced")
G1_left = Q1.unsqueeze(0)

# 第3コアを右直交化
A3 = G3.squeeze(-1)
Q3, R3 = torch.linalg.qr(A3.T, mode="reduced")
G3_right = Q3.T.unsqueeze(-1)

# 左右QRの残差を中心へ吸収
G2_tmp = torch.tensordot(R1, G2, dims=([1], [0]))
G2_center = torch.tensordot(G2_tmp, R3.T, dims=([2], [0]))

X_center2 = reconstruct_tt3(G1_left, G2_center, G3_right)

print("G1_left  :", tuple(G1_left.shape))
print("G2_center:", tuple(G2_center.shape))
print("G3_right :", tuple(G3_right.shape))
print(
    "center-2 reconstruction error =",
    torch.linalg.norm(X_center2 - X_before).item(),
)


G1_left  : (1, 4, 2)
G2_center: (2, 3, 3)
G3_right : (3, 5, 1)
center-2 reconstruction error = 8.426803138067863e-15


## 4. Reduced SVD の shape

中心コアを左展開します。

$$
A
=
G_2^{[C]\langle L\rangle}
\in
\mathbb{R}^{(r_1n_2)\times r_2}.
$$

reduced SVD は、

$$
A
=
U\Sigma V^T.
$$

exact rank を

$$
\rho
=
\operatorname{rank}(A)
$$

と書くと、理論上の有効 shape は

$$
U
\in
\mathbb{R}^{(r_1n_2)\times\rho},
$$

$$
\Sigma
\in
\mathbb{R}^{\rho\times\rho},
$$

$$
V^T
\in
\mathbb{R}^{\rho\times r_2}.
$$

そして、

$$
U^TU=I_\rho,
\qquad
V^TV=I_\rho.
$$

特異値は

$$
\sigma_1
\ge
\sigma_2
\ge
\cdots
\ge
\sigma_\rho
>
0
$$

と並べられます。

今回の数値実装では `full_matrices=False` を使い、**返された SVD 成分をすべて保持します**。

数値的に非常に小さい特異値があっても、この Notebook ではそれを切り捨てません。


## 5. 演習1 — SVD で Center を $2\rightarrow3$ へ移す

### TODO

1. `G2_center` を左展開して行列 `A` を作る
2. `torch.linalg.svd(..., full_matrices=False)` で reduced SVD を行う
3. $U,\Sigma,V^T$ の shape を確認する
4. $U$ を reshape して `G2_left_svd` を作る
5. 
   $$
   \Sigma V^T
   $$
   を作る
6. $\Sigma V^T$ を `G3_right` の左ボンドへ吸収して `G3_center_svd` を作る
7. 新しい TT を再構成する

SVD 後の形は、

$$
X
=
G_1^{[L]}
G_2^{[L]}
G_3^{[C]}
$$

です。

### 実装前に考えること

QR では $R$ を右へ渡しました。

今回は、

$$
\boxed{
\Sigma V^T
}
$$

をまとめて右へ渡します。

$\Sigma$ だけを渡すと $V^T$ による bond 基底変換が失われるため、一般には元の $A$ を復元できません。


In [ ]:
# TODO 1:
# SVD を使って center を 2 -> 3 に移してください。
#
# 1. G2_center を左展開
A = 
#
# 2. reduced SVD
# U, S, Vh = ...
#
# 3. shape を確認
# print("U shape :", ...)
# print("S shape :", ...)
# print("Vh shape:", ...)
#
# 4. U を第2コアへ戻す
# G2_left_svd = ...
#
# 5. Sigma V^T を作る
# Sigma = ...
# transfer = ...
#
# 6. transfer を G3_right の左ボンドへ吸収
# G3_center_svd = ...
#
# 7. 全テンソルを再構成
# X_svd_center3 = ...
#
# print("G2_left_svd shape  :", ...)
# print("G3_center_svd shape:", ...)

print("TODO: exact SVD で center を 2 -> 3 に移す")


## 6. 演習2 — Exact SVD Move の性質を確認する

全ての SVD 成分を保持しているので、この段階では近似を入れていません。

したがって、確認したいのは次の3点です。

### 1. 全テンソル不変性

$$
\|X_{\mathrm{center2}}-X_{\mathrm{SVD,center3}}\|_F
$$

が丸め誤差水準になること。

### 2. 第2コアの左直交性

$U$ を第2コアに戻しているので、

$$
\left(
G_2^{[L]\langle L\rangle}
\right)^T
G_2^{[L]\langle L\rangle}
=
I
$$

になること。

### 3. Center norm

center が第3サイトへ移ったので、

$$
\|X\|_F
=
\|G_3^{[C]}\|_F
$$

になること。

ここまでは QR center move と同じ性質です。


In [ ]:
# TODO 2:
# exact SVD center move の性質を数値確認してください。
#
# 1. 再構成誤差
# reconstruction_error = ...
#
# 2. G2_left_svd の左直交性
# G2_left_matrix = ...
# left_gram = ...
# I_left = ...
# left_orthogonality_error = ...
#
# 3. center norm
# X_norm = ...
# G3_center_norm = ...
# center_norm_error = ...
#
# print("SVD reconstruction error =", ...)
# print("G2 left orthogonality error =", ...)
# print("||X||_F - ||G3_center_svd||_F =", ...)

print("TODO: exact SVD move の再構成・直交性・center norm を確認する")


## 7. $\Sigma$ を Bond 上に残す

exact SVD move では、

$$
G_3^{[C]}
=
(\Sigma V^T)G_3^{[R]}
$$

として、$\Sigma V^T$ をまとめて第3コアへ吸収しました。

しかし SVD の意味を見やすくするため、$\Sigma$ を吸収せずに bond 上へ残すこともできます。

第3コアの左展開を

$$
R_{\mathrm{old}}
=
G_3^{[R]\langle L\rangle}
\in
\mathbb{R}^{r_2\times n_3}
$$

とします。

SVD の右特異ベクトルで bond 基底を回転して、

$$
\widetilde R
=
V^T R_{\mathrm{old}}
\in
\mathbb{R}^{\rho\times n_3}
$$

とします。

これをテンソルへ戻したものを

$$
\widetilde G_3^{[R]}
$$

と書けば、

$$
\boxed{
X
=
G_1^{[L]}
G_2^{[L]}
\Sigma
\widetilde G_3^{[R]}
}
$$

となります。

ここで $\Sigma$ は Tucker の core ではなく、

$$
\boxed{
\text{切断 }2\mid3\text{ の bond 上にある対角な重み}
}
$$

です。


## 8. なぜ $V^T$ を掛けても右直交性が保たれるのか

元の右ブロック行列は、

$$
R_{\mathrm{old}}
R_{\mathrm{old}}^T
=
I_{r_2}
$$

を満たします。

新しい右ブロック行列は

$$
\widetilde R
=
V^TR_{\mathrm{old}}.
$$

したがって、

$$
\begin{aligned}
\widetilde R\widetilde R^T
&=
V^T
R_{\mathrm{old}}
R_{\mathrm{old}}^T
V
\\
&=
V^TI_{r_2}V
\\
&=
V^TV
\\
&=
I_\rho.
\end{aligned}
$$

よって、

$$
\boxed{
\widetilde G_3^{[R]}
\text{ も右直交}
}
$$

です。

つまり SVD は、

- 左側では $U$ によって新しい左直交基底を作る
- 右側では $V$ によって右直交基底を回転する
- その間に $\Sigma$ を残す

という構造を持っています。


## 9. 演習3 — $\Sigma$ を明示した Bond 形式を作る

### TODO

1. `G3_right` を行列 $R_{\mathrm{old}}$ として見る
2. `Vh` を使って
   $$
   \widetilde R
   =
   V^TR_{\mathrm{old}}
   $$
   を作る
3. `R_tilde` を `G3_right_svd` に戻す
4. 右直交性
   $$
   \widetilde R\widetilde R^T=I
   $$
   を確認する
5. 
   $$
   G_1^{[L]}
   G_2^{[L]}
   \Sigma
   \widetilde G_3^{[R]}
   $$
   を収縮して元の $X$ が再構成されることを確認する

### 注意

ここでは $\Sigma$ を第3コアへ吸収しません。

**特異値を bond 上に見える形で残すこと**が目的です。


In [ ]:
# TODO 3:
# Sigma を bond 上に明示した形を作ってください。
#
# 1. G3_right を行列として見る
# R_old = ...
#
# 2. V^T で右 bond 基底を回転
# R_tilde = ...
#
# 3. 3階コアへ戻す
# G3_right_svd = ...
#
# 4. 右直交性を確認
# right_gram = ...
# I_right = ...
# right_orthogonality_error = ...
#
# 5. Sigma を明示したまま X を再構成
# X_sigma_bond = ...
# sigma_bond_reconstruction_error = ...
#
# print("R_tilde shape:", ...)
# print("right orthogonality error =", ...)
# print("Sigma-bond reconstruction error =", ...)

print("TODO: Sigma を bond 上に残した表現を確認する")


## 10. 左右の Schmidt 状態

$\Sigma$ を bond 上に残した形

$$
X
=
G_1^{[L]}
G_2^{[L]}
\Sigma
\widetilde G_3^{[R]}
$$

を考えます。

SVD 後の新しい bond 添字を

$$
\beta
=
1,\ldots,\rho
$$

とします。

### 左状態

第1・第2サイトから、

$$
|L_\beta\rangle
=
\sum_{i_1,i_2,\alpha_1}
G_1^{[L]}(1,i_1,\alpha_1)
G_2^{[L]}(\alpha_1,i_2,\beta)
|i_1\rangle\otimes|i_2\rangle
$$

を定義します。

係数行列としては、

$$
L(i_1,i_2,\beta)
=
\sum_{\alpha_1}
G_1^{[L]}(1,i_1,\alpha_1)
G_2^{[L]}(\alpha_1,i_2,\beta)
$$

です。

### 右状態

第3サイトから、

$$
|R_\beta\rangle
=
\sum_{i_3}
\widetilde G_3^{[R]}(\beta,i_3,1)
|i_3\rangle
$$

を定義します。

係数行列としては、

$$
R(\beta,i_3)
=
\widetilde G_3^{[R]}(\beta,i_3,1)
$$

です。


## 11. 左右の Schmidt 状態が正規直交になる理由

左ブロックについては、

$$
L
=
G_1^{[L]}G_2^{[L]}
$$

と考えます。

$G_1^{[L]}$ と $G_2^{[L]}$ が左直交なので、これまで確認した左ブロックの性質から

$$
\boxed{
L^TL
=
I_\rho
}
$$

です。

したがって、

$$
\langle L_\beta|L_{\beta'}\rangle
=
\delta_{\beta\beta'}.
$$

右ブロックは前節で、

$$
\widetilde R\widetilde R^T
=
I_\rho
$$

を確認しました。

したがって、

$$
\langle R_\beta|R_{\beta'}\rangle
=
\delta_{\beta\beta'}.
$$

つまり、

$$
\boxed{
\{|L_\beta\rangle\}
\text{ と }
\{|R_\beta\rangle\}
\text{ は、それぞれ正規直交系}
}
$$

です。


## 12. 演習4 — 左右 Schmidt 状態の Gram 行列を確認する

### TODO

1. `G1_left` と `G2_left_svd` を収縮して左ブロック `L_block` を作る
2. 物理添字 $(i_1,i_2)$ を一つにまとめ、行列として見る
3. 
   $$
   L^TL
   $$
   を計算する
4. `G3_right_svd` から右ブロック行列 `R_block` を作る
5. 
   $$
   RR^T
   $$
   を計算する
6. それぞれ単位行列との差を Frobenius ノルムで確認する

ここで確認しているのは、**各 Schmidt ラベル $\beta$ が左右で正規直交状態を指定する**ことです。


In [ ]:
# TODO 4:
# 左右 Schmidt 状態の正規直交性を確認してください。
#
# 1. 左ブロック L_beta(i1, i2)
# L_tensor = ...
# L_block = ...
#
# 2. 左 Gram 行列
# left_block_gram = ...
# I_rho_left = ...
# left_block_error = ...
#
# 3. 右ブロック
# R_block = ...
#
# 4. 右 Gram 行列
# right_block_gram = ...
# I_rho_right = ...
# right_block_error = ...
#
# print("L_block shape:", ...)
# print("R_block shape:", ...)
# print("left Schmidt-state orthogonality error =", ...)
# print("right Schmidt-state orthogonality error =", ...)

print("TODO: 左右 Schmidt 状態の Gram 行列を確認する")


## 13. Schmidt 形

ここまでの結果から、

$$
X
=
G_1^{[L]}
G_2^{[L]}
\Sigma
\widetilde G_3^{[R]}
$$

です。

まず、第1・第2サイトをまとめた左ブロックの成分を

$$
L_\beta(i_1,i_2)
=
\sum_{\alpha_1=1}^{r_1}
G_1^{[L]}(1,i_1,\alpha_1)
G_2^{[L]}(\alpha_1,i_2,\beta)
$$

とします。

右ブロックの成分は

$$
R_\beta(i_3)
=
\widetilde G_3^{[R]}(\beta,i_3,1)
$$

です。

すると、$\Sigma$ が対角行列なので、

$$
\begin{aligned}
X(i_1,i_2,i_3)
&=
\sum_{\beta=1}^{\rho}
L_\beta(i_1,i_2)
\sigma_\beta
R_\beta(i_3)
\\
&=
\sum_{\beta=1}^{\rho}
\sigma_\beta
L_\beta(i_1,i_2)
R_\beta(i_3).
\end{aligned}
$$

ここで $\beta$ は各コアの要素を個別に重み付けする添字ではなく、

$$
(i_1,i_2)\mid i_3
$$

という切断に対する、**左ブロック状態と右ブロック状態の共通ラベル**です。

第2 unfolding で見ると、

$$
X^{\langle 2\rangle}
=
L\Sigma R
$$

であり、

$$
\boxed{
X^{\langle 2\rangle}
=
\sum_{\beta=1}^{\rho}
\sigma_\beta
L_{:,\beta}
R_{\beta,:}
}
$$

と書けます。

つまり $\sigma_\beta$ は、第 $\beta$ 番目の直交した rank-1 ブロック成分

$$
L_{:,\beta}R_{\beta,:}
$$

に掛かる係数です。

これを状態ベクトルとして書き直すと、

$$
\boxed{
|X\rangle
=
\sum_{\beta=1}^{\rho}
\sigma_\beta
|L_\beta\rangle
\otimes
|R_\beta\rangle
}
$$

となります。

そして、

$$
\langle L_\beta|L_{\beta'}\rangle
=
\delta_{\beta\beta'},
$$

$$
\langle R_\beta|R_{\beta'}\rangle
=
\delta_{\beta\beta'},
$$

さらに、

$$
\sigma_\beta
\ge
0
$$

です。

したがって、この表現は切断

$$
(i_1,i_2)\mid i_3
$$

に関する **Schmidt 分解**の形になっています。

このとき SVD の特異値

$$
\sigma_\beta
$$

が、その切断における Schmidt 係数です。


## 14. なぜ Schmidt 成分どうしの交差項が消えるのか

テンソル積状態の内積は、

$$
\boxed{
\langle x\otimes y
\mid
x'\otimes y'
\rangle
=
\langle x|x'\rangle
\langle y|y'\rangle
}
$$

です。

したがって、

$$
\begin{aligned}
&
\left(
\langle L_\beta|
\otimes
\langle R_\beta|
\right)
\left(
|L_{\beta'}\rangle
\otimes
|R_{\beta'}\rangle
\right)
\\
&=
\langle L_\beta|L_{\beta'}\rangle
\langle R_\beta|R_{\beta'}\rangle
\\
&=
\delta_{\beta\beta'}
\delta_{\beta\beta'}
\\
&=
\delta_{\beta\beta'}.
\end{aligned}
$$

つまり、

$$
|L_\beta\rangle
\otimes
|R_\beta\rangle
$$

という Schmidt の積状態は、全体空間でも正規直交です。

そのため、

$$
|X\rangle
=
\sum_\beta
\sigma_\beta
|L_\beta\rangle
\otimes
|R_\beta\rangle
$$

のノルムを計算すると、$\beta\neq\beta'$ の交差項はすべて消えます。


## 15. ノルム恒等式

Schmidt 形から、

$$
\begin{aligned}
\|X\|_F^2
&=
\langle X|X\rangle
\\
&=
\sum_{\beta,\beta'}
\sigma_\beta
\sigma_{\beta'}
\langle L_\beta|L_{\beta'}\rangle
\langle R_\beta|R_{\beta'}\rangle.
\end{aligned}
$$

左右の正規直交性を使うと、

$$
\begin{aligned}
\|X\|_F^2
&=
\sum_{\beta,\beta'}
\sigma_\beta
\sigma_{\beta'}
\delta_{\beta\beta'}
\delta_{\beta\beta'}
\\
&=
\sum_{\beta,\beta'}
\sigma_\beta
\sigma_{\beta'}
\delta_{\beta\beta'}
\\
&=
\sum_{\beta=1}^{\rho}
\sigma_\beta^2.
\end{aligned}
$$

ここでは、

$$
\delta_{\beta\beta'}^2
=
\delta_{\beta\beta'}
$$

を使っています。

重要なのは、最初の

$$
\delta_{\beta\beta'}
$$

が左 Schmidt 状態の直交性から、もう一つの

$$
\delta_{\beta\beta'}
$$

が右 Schmidt 状態の直交性から現れることです。

したがって、

$$
\boxed{
\|X\|_F^2
=
\sum_{\beta=1}^{\rho}
\sigma_\beta^2
}
$$

です。

これは SVD の基本式

$$
\|A\|_F^2
=
\sum_\beta\sigma_\beta^2
$$

と、既習の mixed-canonical form のノルム局所化

$$
\|X\|_F
=
\|G_2^{[C]}\|_F
=
\|A\|_F
$$

が同じ内容を表していることも意味します。


## 16. 演習5 — 特異値の二乗和と $\|X\|_F^2$ を比較する

### TODO

SVD で得た特異値を使って、

$$
\sum_\beta\sigma_\beta^2
$$

を計算してください。

そして、

$$
\|X\|_F^2
$$

との差を確認します。

さらに、比較のために

$$
\|G_2^{[C]}\|_F^2
$$

とも比較してください。

確認したい関係は、

$$
\boxed{
\|X\|_F^2
=
\|G_2^{[C]}\|_F^2
=
\sum_\beta\sigma_\beta^2
}
$$

です。

ここでも**特異値は一つも捨てません**。


In [ ]:
# TODO 5:
# ノルム恒等式を数値確認してください。
#
# 1. 全テンソルの Frobenius ノルム二乗
# X_norm_sq = ...
#
# 2. 元の中心コアの Frobenius ノルム二乗
# G2_center_norm_sq = ...
#
# 3. 特異値の二乗和
# singular_value_sq_sum = ...
#
# 4. 差を確認
# center_norm_diff = ...
# svd_norm_diff = ...
#
# print("||X||_F^2 =", ...)
# print("||G2_center||_F^2 =", ...)
# print("sum sigma^2 =", ...)
# print("X vs center difference =", ...)
# print("X vs singular-values difference =", ...)

print("TODO: ||X||_F^2 = sum sigma^2 を確認する")


## 17. QR Center Move と Exact SVD Center Move の比較

| 観点 | QR center move | Exact SVD center move |
| :-- | :-- | :-- |
| 分解対象 | $A=G_2^{[C]\langle L\rangle}$ | 同じ $A$ |
| 分解 | $A=QR$ | $A=U\Sigma V^T$ |
| 第2コアに残す因子 | $Q$ | $U$ |
| 右へ渡す因子 | $R$ | $\Sigma V^T$ |
| 第2コアの左直交性 | ある | ある |
| 全テンソル $X$ | 不変 | 不変 |
| 近似 | なし | 全成分を残せばなし |
| bond の対角重み | 直接には現れない | $\Sigma$ として現れる |
| Schmidt 形 | 直接には読めない | 読める |
| 特異値に基づく圧縮への準備 | 直接はできない | できる |

したがって、

$$
\boxed{
\text{QR は直交化して center を移動する}
}
$$

のに対して、

$$
\boxed{
\text{SVD は直交化して center を移動し、
同時に Schmidt 係数を取り出す}
}
$$

と整理できます。

SVD は QR と別の canonical form を作るというより、

**同じ mixed-canonical 条件を満たしつつ、bond の情報をさらに対角化して見える形にする**

と考えるのが重要です。


## 18. 今回の到達点

この Notebook の流れは、

$$
\boxed{
\begin{aligned}
G_1^{[L]}G_2^{[C]}G_3^{[R]}
&\longrightarrow
G_2^{[C]\langle L\rangle}
=
U\Sigma V^T
\\
&\longrightarrow
G_1^{[L]}G_2^{[L]}
\Sigma
\widetilde G_3^{[R]}
\\
&\longrightarrow
|X\rangle
=
\sum_\beta
\sigma_\beta
|L_\beta\rangle
\otimes
|R_\beta\rangle
\\
&\longrightarrow
\|X\|_F^2
=
\sum_\beta\sigma_\beta^2
\end{aligned}
}
$$

です。

自分の実装で、少なくとも次を確認できれば完了です。

- exact SVD center move 後も全テンソルが不変
- $U$ から作った第2コアが左直交
- $V^T$ で回転した右コアが右直交
- $\Sigma$ を bond 上に残しても同じ $X$ を再構成できる
- 左右 Schmidt 状態の Gram 行列が単位行列になる
- 
  $$
  \|X\|_F^2
  =
  \sum_\beta\sigma_\beta^2
  $$
  が数値的にも成立する

### 次の Notebook

次は独立した新しい学習単位として、

$$
\boxed{
\text{truncated SVD と rank truncation}
}
$$

へ進みます。

この Notebook 08 では、**特異値を捨てる処理は行いません。**
